In [0]:
"""
id: source_0d6b962c
template: source
templateVersion: 3.0.0
name: Nation geography
position:
  x: 0
  y: 435
description:
  text: Read data from a specified table source with optional metric view selection or direct query.
  hash: 40c50091
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792978866453
config:
  table_source:
    tableName: "`:catalog || '`.bronze.`nations_raw'`"
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "`:catalog || '`.bronze.`nations_raw'`"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_0d6b962c.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_0d6b962c" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_0d6b962c.data"])
if "source_0d6b962c" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_0d6b962c", "counts": {"data": int(ctx["source_0d6b962c.data"].count())}}}, raw=True)

In [0]:
"""
id: source_8ff54416
template: source
templateVersion: 3.0.0
name: Order headers
position:
  x: 0
  y: 145
description:
  text: Read data from the specified table.
  hash: 16eafe64
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792978866453
config:
  table_source:
    tableName: "`:catalog || '`.silver.`order_headers'`"
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "`:catalog || '`.silver.`order_headers'`"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_8ff54416.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_8ff54416" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_8ff54416.data"])
if "source_8ff54416" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_8ff54416", "counts": {"data": int(ctx["source_8ff54416.data"].count())}}}, raw=True)

In [0]:
"""
id: source_bdf272d6
template: source
templateVersion: 3.0.0
name: Order lines
position:
  x: 0
  y: 0
description:
  text: Read from a specified table.
  hash: 89c44136
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792978866452
config:
  table_source:
    tableName: "`:catalog || '`.silver.`order_items'`"
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "`:catalog || '`.silver.`order_items'`"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_bdf272d6.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_bdf272d6" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_bdf272d6.data"])
if "source_bdf272d6" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_bdf272d6", "counts": {"data": int(ctx["source_bdf272d6.data"].count())}}}, raw=True)

In [0]:
"""
id: source_f342422e
template: source
templateVersion: 3.0.0
name: Customer history
position:
  x: 0
  y: 290
description:
  text: Read data from a specified table, optionally using a metric view selection.
  hash: 662621ca
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792978866453
config:
  table_source:
    tableName: "`:catalog || '`.silver.`customer_versions'`"
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "`:catalog || '`.silver.`customer_versions'`"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_f342422e.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_f342422e" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_f342422e.data"])
if "source_f342422e" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_f342422e", "counts": {"data": int(ctx["source_f342422e.data"].count())}}}, raw=True)

In [0]:
"""
id: join_268ac8d9
template: join
templateVersion: 3.0.0
name: Header line join
position:
  x: 260
  y: 72
description:
  text: Perform an inner join between two tables on the order_key column, returning matched rows along with unmatched rows from each table separately.
  hash: 844edca0
codegenHash: ea5821d113ebbb3b
updatedAt: 1792978866452
config:
  join_type: inner
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: order_key
          right: order_key
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
checkpoint:
  mode: auto
input:
  - node: source_bdf272d6
    input_port: left
    output_port: data
  - node: source_8ff54416
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "order_key",
                    "right": "order_key"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["source_bdf272d6.data"],
    "right": ctx["source_8ff54416.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_268ac8d9.joined_data"] = out["joined_data"]
ctx["join_268ac8d9.left_unmatched"] = out["left_unmatched"]
ctx["join_268ac8d9.right_unmatched"] = out["right_unmatched"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["join_268ac8d9.joined_data"], "localCheckpoint", None)) and callable(getattr(ctx["join_268ac8d9.left_unmatched"], "localCheckpoint", None)) and callable(getattr(ctx["join_268ac8d9.right_unmatched"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["join_268ac8d9.joined_data"]
        _lb_cp_original_1 = ctx["join_268ac8d9.left_unmatched"]
        _lb_cp_original_2 = ctx["join_268ac8d9.right_unmatched"]
        try:
            ctx["join_268ac8d9.joined_data"] = ctx["join_268ac8d9.joined_data"].localCheckpoint(eager=True)
            ctx["join_268ac8d9.left_unmatched"] = ctx["join_268ac8d9.left_unmatched"].localCheckpoint(eager=True)
            ctx["join_268ac8d9.right_unmatched"] = ctx["join_268ac8d9.right_unmatched"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["join_268ac8d9.joined_data"] is not _lb_cp_original_0:
                try:
                    ctx["join_268ac8d9.joined_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_268ac8d9.joined_data"] = _lb_cp_original_0
            if ctx["join_268ac8d9.left_unmatched"] is not _lb_cp_original_1:
                try:
                    ctx["join_268ac8d9.left_unmatched"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_268ac8d9.left_unmatched"] = _lb_cp_original_1
            if ctx["join_268ac8d9.right_unmatched"] is not _lb_cp_original_2:
                try:
                    ctx["join_268ac8d9.right_unmatched"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_268ac8d9.right_unmatched"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "join_268ac8d9" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_268ac8d9.joined_data"])
    display(ctx["join_268ac8d9.left_unmatched"])
    display(ctx["join_268ac8d9.right_unmatched"])
if "join_268ac8d9" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_268ac8d9", "counts": {"joined_data": int(ctx["join_268ac8d9.joined_data"].count()), "left_unmatched": int(ctx["join_268ac8d9.left_unmatched"].count()), "right_unmatched": int(ctx["join_268ac8d9.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: sql_ba47f3bb
template: sql
templateVersion: 1.0.0
name: Customer version at order time
position:
  x: 520
  y: 120
description:
  text: Join order and customer history on matching keys and date ranges to enrich data with market segment and customer nation.
  hash: 2429fca0
codegenHash: 93d0fe419523af76
updatedAt: 1792978884373
config:
  query: |
    SELECT j.*, h.market_segment AS market_segment_at_order_time,
      h.customer_nation_key
    FROM Header line join j
    JOIN Customer history h
      ON j.customer_key=h.customer_key
     AND j.order_date>=h.__START_AT
     AND (h.__END_AT IS NULL OR j.order_date<h.__END_AT)
checkpoint:
  mode: auto
input:
  - node: join_268ac8d9
    input_port: data
    output_port: joined_data
  - node: source_f342422e
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT j.*, h.market_segment AS market_segment_at_order_time,\n  h.customer_nation_key\nFROM Header line join j\nJOIN Customer history h\n  ON j.customer_key=h.customer_key\n AND j.order_date>=h.__START_AT\n AND (h.__END_AT IS NULL OR j.order_date<h.__END_AT)\n"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["join_268ac8d9.joined_data"],
        ctx["source_f342422e.data"]
    ],
    "data__sources": [
        {
            "node": "join_268ac8d9",
            "output_port": "joined_data",
            "name": "Header line join",
            "df_name": "Header_line_join"
        },
        {
            "node": "source_f342422e",
            "output_port": "data",
            "name": "Customer history",
            "df_name": "Customer_history"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sql_ba47f3bb.result"] = out["result"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["sql_ba47f3bb.result"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["sql_ba47f3bb.result"]
        try:
            ctx["sql_ba47f3bb.result"] = ctx["sql_ba47f3bb.result"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["sql_ba47f3bb.result"] is not _lb_cp_original_0:
                try:
                    ctx["sql_ba47f3bb.result"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["sql_ba47f3bb.result"] = _lb_cp_original_0
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "sql_ba47f3bb" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sql_ba47f3bb.result"])
if "sql_ba47f3bb" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sql_ba47f3bb", "counts": {"result": int(ctx["sql_ba47f3bb.result"].count())}}}, raw=True)

In [0]:
"""
id: join_9bf9b30b
template: join
templateVersion: 3.0.0
name: Nation geography join
position:
  x: 780
  y: 180
description:
  text: Perform an inner join between two tables on matching nation key columns, showing matched records and unmatched from each table separately.
  hash: 743f72aa
codegenHash: ea5821d113ebbb3b
updatedAt: 1792978894234
config:
  join_type: inner
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: customer_nation_key
          right: n_nationkey
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
checkpoint:
  mode: auto
input:
  - node: sql_ba47f3bb
    input_port: left
    output_port: result
  - node: source_0d6b962c
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "customer_nation_key",
                    "right": "n_nationkey"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["sql_ba47f3bb.result"],
    "right": ctx["source_0d6b962c.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_9bf9b30b.joined_data"] = out["joined_data"]
ctx["join_9bf9b30b.left_unmatched"] = out["left_unmatched"]
ctx["join_9bf9b30b.right_unmatched"] = out["right_unmatched"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["join_9bf9b30b.joined_data"], "localCheckpoint", None)) and callable(getattr(ctx["join_9bf9b30b.left_unmatched"], "localCheckpoint", None)) and callable(getattr(ctx["join_9bf9b30b.right_unmatched"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["join_9bf9b30b.joined_data"]
        _lb_cp_original_1 = ctx["join_9bf9b30b.left_unmatched"]
        _lb_cp_original_2 = ctx["join_9bf9b30b.right_unmatched"]
        try:
            ctx["join_9bf9b30b.joined_data"] = ctx["join_9bf9b30b.joined_data"].localCheckpoint(eager=True)
            ctx["join_9bf9b30b.left_unmatched"] = ctx["join_9bf9b30b.left_unmatched"].localCheckpoint(eager=True)
            ctx["join_9bf9b30b.right_unmatched"] = ctx["join_9bf9b30b.right_unmatched"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["join_9bf9b30b.joined_data"] is not _lb_cp_original_0:
                try:
                    ctx["join_9bf9b30b.joined_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_9bf9b30b.joined_data"] = _lb_cp_original_0
            if ctx["join_9bf9b30b.left_unmatched"] is not _lb_cp_original_1:
                try:
                    ctx["join_9bf9b30b.left_unmatched"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_9bf9b30b.left_unmatched"] = _lb_cp_original_1
            if ctx["join_9bf9b30b.right_unmatched"] is not _lb_cp_original_2:
                try:
                    ctx["join_9bf9b30b.right_unmatched"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_9bf9b30b.right_unmatched"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "join_9bf9b30b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_9bf9b30b.joined_data"])
    display(ctx["join_9bf9b30b.left_unmatched"])
    display(ctx["join_9bf9b30b.right_unmatched"])
if "join_9bf9b30b" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_9bf9b30b", "counts": {"joined_data": int(ctx["join_9bf9b30b.joined_data"].count()), "left_unmatched": int(ctx["join_9bf9b30b.left_unmatched"].count()), "right_unmatched": int(ctx["join_9bf9b30b.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: filter_1ec3716b
template: filter
templateVersion: 2.0.0
name: Segment filter
position:
  x: 1040
  y: 190
description:
  text: Keep rows that match the condition and separate those that don't.
  hash: 4b80f482
codegenHash: c63d5f6ab0675849
updatedAt: 1792978903860
config:
  condition: :segment = 'ALL' OR market_segment_at_order_time = :segment
checkpoint:
  mode: auto
input:
  - node: join_9bf9b30b
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.expr(condition)
    return {
        "filtered_data": df.filter(keep),
        "excluded_data": df.filter(~F.coalesce(keep, F.lit(False))),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": ":segment = 'ALL' OR market_segment_at_order_time = :segment"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_9bf9b30b.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_1ec3716b.filtered_data"] = out["filtered_data"]
ctx["filter_1ec3716b.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_1ec3716b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_1ec3716b.filtered_data"])
    display(ctx["filter_1ec3716b.excluded_data"])
if "filter_1ec3716b" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_1ec3716b", "counts": {"filtered_data": int(ctx["filter_1ec3716b.filtered_data"].count()), "excluded_data": int(ctx["filter_1ec3716b.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: aggregate_10dee4ef
template: aggregate
templateVersion: 2.0.0
name: Customer totals
position:
  x: 1300
  y: 210
description:
  text: Group data by customer and market segment keys, then calculate total net revenue and count distinct orders.
  hash: f83a9819
codegenHash: 39e07f1e5701539d
updatedAt: 1792978913987
config:
  group_bys:
    - expr: customer_key
      type: expr
    - expr: market_segment_at_order_time
      type: expr
    - expr: customer_nation_key
      type: expr
    - expr: n_regionkey
      type: expr
  aggregations:
    - columnExpr:
        expr: extended_price * (1 - discount_rate)
        type: expr
      fn: SUM
      alias: net_revenue
    - columnExpr:
        expr: order_key
        type: expr
      fn: COUNT_DISTINCT
      alias: order_count
checkpoint:
  mode: auto
input:
  - node: filter_1ec3716b
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def _quoted_column(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _column_name(column_expr: Dict[str, Any], df) -> str:
    name = column_expr.get("expr", "")
    if column_expr.get("type") == "column" and name != "*" and name in df.columns:
        return _quoted_column(name)
    return name

def _column_arg(column_expr: Dict[str, Any], df):
    name = column_expr.get("expr", "")
    if column_expr.get("type") == "expr":
        return F.expr(name)
    return F.col(_column_name(column_expr, df))

def _sql_arg(column_expr: Dict[str, Any], df) -> str:
    return _column_name(column_expr, df)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(_column_arg(col_expr, df))
        elif fn == "-" or fn == "_":
            col = _column_arg(col_expr, df)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({_sql_arg(col_expr, df)}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            col = F.concat_ws(sep, F.collect_list(_column_arg(col_expr, df)))
        elif fn == "COUNT_DISTINCT":
            arg = _column_arg(col_expr, df) if col_expr.get("type") == "column" else F.expr(raw_expr)
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({_sql_arg(col_expr, df)})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        _column_arg(gb, df) for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "customer_key",
            "type": "expr"
        },
        {
            "expr": "market_segment_at_order_time",
            "type": "expr"
        },
        {
            "expr": "customer_nation_key",
            "type": "expr"
        },
        {
            "expr": "n_regionkey",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "extended_price * (1 - discount_rate)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "net_revenue",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "order_key",
                "type": "expr"
            },
            "fn": "COUNT_DISTINCT",
            "alias": "order_count",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_1ec3716b.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_10dee4ef.aggregated_data"] = out["aggregated_data"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["aggregate_10dee4ef.aggregated_data"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["aggregate_10dee4ef.aggregated_data"]
        try:
            ctx["aggregate_10dee4ef.aggregated_data"] = ctx["aggregate_10dee4ef.aggregated_data"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["aggregate_10dee4ef.aggregated_data"] is not _lb_cp_original_0:
                try:
                    ctx["aggregate_10dee4ef.aggregated_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["aggregate_10dee4ef.aggregated_data"] = _lb_cp_original_0
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "aggregate_10dee4ef" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_10dee4ef.aggregated_data"])
if "aggregate_10dee4ef" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "aggregate_10dee4ef", "counts": {"aggregated_data": int(ctx["aggregate_10dee4ef.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: sql_6c0919d8
template: sql
templateVersion: 1.0.0
name: Rank within segment
position:
  x: 1560
  y: 230
description:
  text: Add a rank within each market segment based on revenue and customer key.
  hash: 7f52a0ca
codegenHash: 93d0fe419523af76
updatedAt: 1792978923108
config:
  query: |
    SELECT *, row_number() OVER (
      PARTITION BY market_segment_at_order_time
      ORDER BY net_revenue DESC, customer_key
    ) AS rank_in_segment
    FROM Customer totals
checkpoint:
  mode: auto
input:
  - node: aggregate_10dee4ef
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT *, row_number() OVER (\n  PARTITION BY market_segment_at_order_time\n  ORDER BY net_revenue DESC, customer_key\n) AS rank_in_segment\nFROM Customer totals\n"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["aggregate_10dee4ef.aggregated_data"]
    ],
    "data__sources": [
        {
            "node": "aggregate_10dee4ef",
            "output_port": "aggregated_data",
            "name": "Customer totals",
            "df_name": "Customer_totals"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sql_6c0919d8.result"] = out["result"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["sql_6c0919d8.result"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["sql_6c0919d8.result"]
        try:
            ctx["sql_6c0919d8.result"] = ctx["sql_6c0919d8.result"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["sql_6c0919d8.result"] is not _lb_cp_original_0:
                try:
                    ctx["sql_6c0919d8.result"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["sql_6c0919d8.result"] = _lb_cp_original_0
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "sql_6c0919d8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sql_6c0919d8.result"])
if "sql_6c0919d8" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sql_6c0919d8", "counts": {"result": int(ctx["sql_6c0919d8.result"].count())}}}, raw=True)

In [0]:
"""
id: filter_97684a2b
template: filter
templateVersion: 2.0.0
name: Top ten filter
position:
  x: 1820
  y: 245
description:
  text: Keep rows where rank_in_segment is 10 or less, separate the rest.
  hash: 48701bf4
codegenHash: c63d5f6ab0675849
updatedAt: 1792978932507
config:
  condition: rank_in_segment <= 10
checkpoint:
  mode: auto
input:
  - node: sql_6c0919d8
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.expr(condition)
    return {
        "filtered_data": df.filter(keep),
        "excluded_data": df.filter(~F.coalesce(keep, F.lit(False))),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "rank_in_segment <= 10"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sql_6c0919d8.result"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_97684a2b.filtered_data"] = out["filtered_data"]
ctx["filter_97684a2b.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_97684a2b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_97684a2b.filtered_data"])
    display(ctx["filter_97684a2b.excluded_data"])
if "filter_97684a2b" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_97684a2b", "counts": {"filtered_data": int(ctx["filter_97684a2b.filtered_data"].count()), "excluded_data": int(ctx["filter_97684a2b.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: sort_1acf9e6b
template: sort
templateVersion: 1.0.0
name: Segment and rank sort
position:
  x: 2080
  y: 260
description:
  text: Sort data by market segment and rank in ascending order.
  hash: d0bb1c26
previewCodeHash: 9a8851020e351e26
codegenHash: 8d8595d72d737d1d
updatedAt: 1792978941116
config:
  sort_expressions:
    - columnExpr:
        expr: market_segment_at_order_time
        type: expr
      sortBy: ASC
    - columnExpr:
        expr: rank_in_segment
        type: expr
      sortBy: ASC
checkpoint:
  mode: auto
input:
  - node: filter_97684a2b
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "market_segment_at_order_time",
                "type": "expr"
            },
            "sortBy": "ASC"
        },
        {
            "columnExpr": {
                "expr": "rank_in_segment",
                "type": "expr"
            },
            "sortBy": "ASC"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_97684a2b.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sort_1acf9e6b.sorted_data"] = out["sorted_data"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["sort_1acf9e6b.sorted_data"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["sort_1acf9e6b.sorted_data"]
        try:
            ctx["sort_1acf9e6b.sorted_data"] = ctx["sort_1acf9e6b.sorted_data"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["sort_1acf9e6b.sorted_data"] is not _lb_cp_original_0:
                try:
                    ctx["sort_1acf9e6b.sorted_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["sort_1acf9e6b.sorted_data"] = _lb_cp_original_0
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "sort_1acf9e6b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sort_1acf9e6b.sorted_data"])
if "sort_1acf9e6b" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sort_1acf9e6b", "counts": {"sorted_data": int(ctx["sort_1acf9e6b.sorted_data"].count())}}}, raw=True)

In [0]:
"""
id: output_b1498382
template: output
templateVersion: 4.0.0
name: "Gold output: Top customers"
position:
  x: 2340
  y: 275
description:
  text: Overwrite the table vdp_top_customers in the gold schema with the input data.
  hash: 5b80326f
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792979145686
config:
  output_type: table
  catalog: "`:catalog`"
  schema: gold
  table_name: vdp_top_customers
  write_mode: overwrite
checkpoint:
  mode: auto
input:
  - node: sort_1acf9e6b
    input_port: data
    output_port: sorted_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "`:catalog`",
    "schema": "gold",
    "table_name": "vdp_top_customers",
    "write_mode": "overwrite",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sort_1acf9e6b.sorted_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM `:catalog`.gold.vdp_top_customers").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_b1498382.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_b1498382" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_b1498382.result"])
if "output_b1498382" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_b1498382", "counts": {"result": int(ctx["output_b1498382.result"].count())}}}, raw=True)

In [0]:
"""
id: visualization_03a975c0
template: visualization
templateVersion: 1.0.0
name: Top customers visualization
position:
  x: 2600
  y: 275
previewCodeHash: 380b980dd1ba55dc
codegenHash: 95fa3a0d85544146
updatedAt: 1792979176415
config:
  editorSpec:
    renderSpec:
      version: 3
      widgetType: bar
      encodings:
        x:
          fieldName: customer_key
          scale:
            type: categorical
          displayName: customer_key
        y:
          fieldName: net revenue
          scale:
            type: quantitative
          displayName: net revenue
        color:
          fieldName: market_segment_at_order_time
          scale:
            type: quantitative
          displayName: segment
      frame:
        title: Top ten customers by segment
        showTitle: true
    queries:
      - queryName: main_query
        query:
          datasetName: lakeflow_designer_upstream
          fields:
            - fieldName: customer_key
              expression: "`customer_key`"
            - fieldName: net revenue
              expression: SUM(`net_revenue`)
            - fieldName: market_segment_at_order_time
              expression: "`market_segment_at_order_time`"
          filters: []
          disaggregatedData: false
    parameterQueries: []
    version: 2
  sql_query: |-
    WITH q AS (SELECT * FROM lakeflow_designer_upstream)
    SELECT `customer_key` AS customer_key, SUM(`net_revenue`) AS `net revenue`, `market_segment_at_order_time` AS market_segment_at_order_time
    FROM q
    GROUP BY customer_key, market_segment_at_order_time
checkpoint:
  mode: auto
input:
  - node: sort_1acf9e6b
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Any, Dict

_SOURCE_VIEW = "lakeflow_designer_upstream"

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        return {"data": None}

    sql_query = config.get("sql_query")
    if not isinstance(sql_query, str) or not sql_query.strip():
        return {"data": df}

    try:
        df.createOrReplaceTempView(_SOURCE_VIEW)
        globals().setdefault("_lb_views", set()).add(_SOURCE_VIEW)
        return {"data": spark.sql(sql_query)}
    except Exception as exc:
        print("[visualization] compiled query failed, passing input through:", exc, sql_query)
        return {"data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "editorSpec": {
        "renderSpec": {
            "version": 3,
            "widgetType": "bar",
            "encodings": {
                "x": {
                    "fieldName": "customer_key",
                    "scale": {
                        "type": "categorical",
                        "sort": None
                    },
                    "displayName": "customer_key",
                    "axis": None
                },
                "y": {
                    "fieldName": "net revenue",
                    "scale": {
                        "type": "quantitative"
                    },
                    "displayName": "net revenue",
                    "format": None,
                    "axis": None
                },
                "color": {
                    "fieldName": "market_segment_at_order_time",
                    "scale": {
                        "type": "quantitative"
                    },
                    "displayName": "segment"
                },
                "extra": None,
                "label": None
            },
            "frame": {
                "title": "Top ten customers by segment",
                "showTitle": True
            }
        },
        "queries": [
            {
                "queryName": "main_query",
                "query": {
                    "datasetName": "lakeflow_designer_upstream",
                    "fields": [
                        {
                            "fieldName": "customer_key",
                            "expression": "`customer_key`"
                        },
                        {
                            "fieldName": "net revenue",
                            "expression": "SUM(`net_revenue`)"
                        },
                        {
                            "fieldName": "market_segment_at_order_time",
                            "expression": "`market_segment_at_order_time`"
                        }
                    ],
                    "filters": [],
                    "disaggregatedData": False
                }
            }
        ],
        "parameterQueries": [],
        "version": 2
    },
    "sql_query": "WITH q AS (SELECT * FROM lakeflow_designer_upstream)\nSELECT `customer_key` AS customer_key, SUM(`net_revenue`) AS `net revenue`, `market_segment_at_order_time` AS market_segment_at_order_time\nFROM q\nGROUP BY customer_key, market_segment_at_order_time"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sort_1acf9e6b.sorted_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["visualization_03a975c0.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "visualization_03a975c0" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["visualization_03a975c0.data"])
if "visualization_03a975c0" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "visualization_03a975c0", "counts": {"data": int(ctx["visualization_03a975c0.data"].count())}}}, raw=True)